# Rankine vortex

### Philipp Schlatter, LSTM-FAU, 2026

In [4]:
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import collections
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

twopi = 2 * np.pi

# ------------------------------------------------------------------
# Rankine vortex parameters
# ------------------------------------------------------------------
circulation = 10e-3      # Γ [m^2/s]
core_radius = 0.05       # a [m] — radius of solid-body core

# ------------------------------------------------------------------
# Particle / marker parameters
# ------------------------------------------------------------------
rmax  = 0.2
rmark = 0.003            # size of each cross marker
N     = 600              # number of particles

# radial coordinate — sqrt-uniform so particles are uniform in 2D area
part_r = rmax * np.sqrt(np.linspace(0, 1, N + 1)[1:])

# angular coordinate & marker rotation
np.random.seed(42)
part_th  = twopi * np.random.rand(N)
part_rot = np.zeros(N)

# ------------------------------------------------------------------
# Rankine vortex kinematics (steady — no viscosity needed)
# ------------------------------------------------------------------
inside = part_r <= core_radius

# d(theta)/dt = v_theta / r
dth_dt = np.empty_like(part_r)
dth_dt[inside]  = circulation / (twopi * core_radius**2)          # solid body
dth_dt[~inside] = circulation / (twopi * part_r[~inside]**2)      # potential vortex

# Marker rotation rate = ½ × vorticity
drot_dt = np.empty_like(part_r)
drot_dt[inside]  = circulation / (twopi * core_radius**2)
drot_dt[~inside] = 0.0

# ------------------------------------------------------------------
# Time-stepping control
# ------------------------------------------------------------------
dt         = 0.01
frame_skip = 10          # render every 10th physics step
t_max      = 25.0

# ------------------------------------------------------------------
# Figure setup
# ------------------------------------------------------------------
fig = plt.figure(figsize=(6, 6))
ax  = plt.axes((0, 0, 1, 1))
ax.set_xlim(-0.7 * rmax, 0.7 * rmax)
ax.set_ylim(-0.7 * rmax, 0.7 * rmax)
ax.set_aspect('equal')
ax.set_axis_off()

# --- RED CORE CIRCLE ------------------------------------------------
core_circle = plt.Circle(
    (0, 0), core_radius,
    color='red',
    fill=False,
    linewidth=1.,
    linestyle='-',
    zorder=5
)
ax.add_patch(core_circle)
# ------------------------------------------------------------------

linesegs = np.empty((2 * N, 2, 2))
linecoll = collections.LineCollection(
    linesegs,
    linewidths=1,
    colors=((0.0, 0.0, 0.0), (0.75, 0.75, 0.75)),
    linestyle='solid'
)
ax.add_collection(linecoll)

time_text = fig.text(0.01, 0.01, 't = 0.00 s', fontsize='x-large',
                     bbox=dict(facecolor='w', alpha=1))

# Mutable state container so update() can modify arrays in-place
state = {'th': part_th.copy(), 'rot': part_rot.copy()}

def init():
    """Reset animation to initial condition."""
    state['th']  = part_th.copy()
    state['rot'] = part_rot.copy()
    linecoll.set_segments(np.empty((0, 2, 2)))
    time_text.set_text('t = 0.00 s')
    # Ensure the red circle stays visible on reset
    if core_circle not in ax.patches:
        ax.add_patch(core_circle)
    return linecoll, time_text, core_circle

def update(frame):
    """Advance physics and redraw markers."""
    for _ in range(frame_skip):
        state['th']  += dt * dth_dt
        state['rot'] += dt * drot_dt

    t = frame * dt * frame_skip

    # particle positions
    x = part_r * np.cos(state['th'])
    y = part_r * np.sin(state['th'])

    # marker orientation
    cr = rmark * np.cos(state['rot'])
    sr = rmark * np.sin(state['rot'])

    # first line of each cross (black)
    linesegs[0::2, 0, 0] = x + cr
    linesegs[0::2, 0, 1] = y + sr
    linesegs[0::2, 1, 0] = x - cr
    linesegs[0::2, 1, 1] = y - sr

    # second line of each cross (grey, perpendicular)
    linesegs[1::2, 0, 0] = x - sr
    linesegs[1::2, 0, 1] = y + cr
    linesegs[1::2, 1, 0] = x + sr
    linesegs[1::2, 1, 1] = y - cr

    linecoll.set_segments(linesegs)
    time_text.set_text(f't = {t:.02f} s')
    return linecoll, time_text, core_circle

# ------------------------------------------------------------------
# Build animation
# ------------------------------------------------------------------
n_frames = int(t_max / (dt * frame_skip))
anim = FuncAnimation(fig, update, frames=n_frames, init_func=init,
                     blit=True, interval=50)

# ------------------------------------------------------------------
# 1) Save to MP4 (requires ffmpeg)
# ------------------------------------------------------------------
anim.save('rankine_vortex.mp4', writer='ffmpeg', fps=20, dpi=150)
print("Saved rankine_vortex.mp4")

# ------------------------------------------------------------------
# 2) Display interactively in Jupyter as HTML5 video
# ------------------------------------------------------------------
display(HTML(anim.to_html5_video()))
plt.close(fig)   # prevent the static final frame from appearing below

Saved rankine_vortex.mp4
